# 02 · Data Cleaning

| | |
|---|---|
| **Project** | Mental Health & Suicide Prevention Data Analysis (Canada) |
| **Pipeline step** | 2 of 10 |
| **Author** | Fatima and Danny |
| **Status** | ✅ Team reviewed and agreed |
| **Date** | 2026-09-01 |
| **Audit report** | `reports/Report 3 - Data Cleaning and Bug Fixes.md` |

---

## Overview

This notebook cleans all eight raw datasets and writes tidy outputs to `data/processed/02_cleaned/`.

**Datasets processed**

| Key | Source | Cleaner |
|---|---|---|
| `perceived_mh_annual` | StatCan 13-10-0972 | `clean_statcan_long` |
| `suicidal_thoughts` | StatCan catalogue | `clean_statcan_long` |
| `stress_coping` | StatCan catalogue | `clean_statcan_long` |
| `perceived_health_quarterly` | StatCan catalogue | `clean_statcan_long` |
| `cchs_mh_disorders` | StatCan catalogue (160 992 rows) | `clean_statcan_long` |
| `cihi_mh_services` | CIHI chart-config CSV | `clean_cihi_vizconfig` |
| `cihi_children_youth` | CIHI Excel workbook (2 sheets) | `clean_cihi_children_youth` |
| `mhacs_2022_pumf` | StatCan MHACS 2022 PUMF microdata | `clean_mhacs_pumf` |

**Notebook structure**

| Section | Purpose |
|---|---|
| §1 Libraries | Import dependencies |
| §2 Paths | Resolve `data/raw` and `data/processed` roots |
| §3 Load | Load all raw datasets into memory |
| §4 Cleaning functions | Define all four cleaning functions |
| §5 Run pipeline | Apply cleaners and write processed CSVs |


## §1 · Libraries

Standard library and third-party imports used throughout the notebook.
`re` is required by `clean_cihi_children_youth` for dynamic year-column detection.


In [1]:
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.5 | numpy 2.5.2


## §2 · Paths

`find_root` walks up the directory tree until it locates `data/raw/`, so the notebook runs correctly whether launched from the repo root or from the `notebooks/` folder.


In [2]:
def find_root(start: Path) -> Path:
    """Walk up until we find the folder that contains data/raw (works from repo root or /notebooks)."""
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("Could not find data/raw above " + str(start))

ROOT = find_root(Path.cwd())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print("root     :", ROOT)
print("raw      :", RAW)
print("processed:", PROCESSED)


root     : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-
raw      : /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/raw
processed: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed


## §3 · Load raw datasets

All eight raw files are registered in `DATASETS`. StatCan CSVs use `utf-8-sig` encoding (byte-order mark). The CIHI Excel workbook contains two machine-readable hidden sheets (`Table8DATA_to hide`, `Table13DATA_to hide`); these are parsed with `header=None` and reconstructed using row 1 as the column header.


In [3]:
DATASETS = {
    "perceived_mh_annual":        {"file": "StatCan 13-10-0972 – perceived mental health.csv",                                              "kind": "statcan_long"},
    "suicidal_thoughts":          {"file": "Catalogue Entry Mental health characteristics and suicidal thoughts.csv",                        "kind": "statcan_long"},
    "stress_coping":              {"file": "Catalogue Entry Mental health characteristics Ability to handle stress and sources of stress.csv","kind": "statcan_long"},
    "perceived_health_quarterly": {"file": "Catalogue Entry Mental health indicators.csv",                                                   "kind": "statcan_long"},
    "cchs_mh_disorders":          {"file": "Catalogue Entry Perceived health, by gender and province.csv",                                   "kind": "statcan_long"},
    "cihi_mh_services":           {"file": "health services for mental illness and alcoholdrug induced disorders.csv",                       "kind": "cihi_vizconfig"},
    "cihi_children_youth":        {"file": "care-children-youth-with-mental-disorders-data-tables-en.xlsx",                                  "kind": "excel_multitable"},
    "mhacs_2022_pumf":            {"file": "MHACS 2022 Public Use Microdata.csv",                                                            "kind": "microdata"},
}


def load_dataset(key: str) -> pd.DataFrame:
    """Load a single CSV dataset from data/raw using the DATASETS registry."""
    spec = DATASETS[key]
    path = RAW / spec["file"]
    if spec["kind"] in ("statcan_long", "cihi_vizconfig"):
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if spec["kind"] == "microdata":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"{key}: kind={spec['kind']} is loaded separately (see Excel block below)")


# Load all CSV-backed datasets
raw = {k: load_dataset(k) for k, v in DATASETS.items() if v["kind"] != "excel_multitable"}

# Load the CIHI Excel workbook: parse only the two hidden data sheets.
# Row 0 is a title row; row 1 contains the column headers.
_xls = pd.ExcelFile(RAW / DATASETS["cihi_children_youth"]["file"])
raw_excel = {}
for _sheet in [s for s in _xls.sheet_names if s.endswith("_to hide")]:
    _t    = _xls.parse(_sheet, header=None)
    _body = _t.iloc[2:].reset_index(drop=True)
    _body.columns = [str(h).replace("\n", " ").strip() for h in _t.iloc[1]]
    raw_excel[_sheet] = _body

# Report shapes
for k, df in raw.items():
    print(f"{k:28} {df.shape}")
for k, df in raw_excel.items():
    print(f"{k:28} {df.shape}  (excel)")


perceived_mh_annual          (936, 18)
suicidal_thoughts            (8208, 18)
stress_coping                (27360, 18)
perceived_health_quarterly   (6318, 17)
cchs_mh_disorders            (160992, 18)
cihi_mh_services             (264, 14)
mhacs_2022_pumf              (9861, 602)
Table8DATA_to hide           (216, 13)  (excel)
Table13DATA_to hide          (216, 13)  (excel)


## §4 · Cleaning functions

The four functions below cover every dataset in the pipeline. All bugs and warnings identified during the team code review (Report 3, 2026-09-01) are resolved.

| Function | Datasets | What it produces |
|---|---|---|
| `clean_statcan_long` | 5 StatCan CSVs | Tidy long table; `Percent` / `Low 95%` / `High 95%` characteristics pivoted to `value`, `ci_low`, `ci_high` columns |
| `clean_cihi_vizconfig` | `cihi_mh_services` | One row per x/y pair: `indicator`, `breakdown`, `group`, `value`, `ci_low`, `ci_high` |
| `clean_cihi_children_youth` | CIHI Excel (2 sheets) | Long format; CI strings parsed to `ci_low`/`ci_high`; point estimate preserved in `value` |
| `clean_mhacs_pumf` | MHACS 2022 PUMF | Missing-value codes replaced with `NaN` for all numeric D-prefix variables |


In [4]:
def clean_statcan_long(df: pd.DataFrame) -> pd.DataFrame:
    """Clean a StatCan long-format CSV.

    Steps
    -----
    1. Normalise column names (lowercase, strip whitespace; rename legacy names).
    2. Parse REF_DATE into ref_date_raw (original string) and start_year (Int64).
    3. Convert VALUE to float; multiply by 1000 where SCALAR_FACTOR == 'thousands'.
    4. Normalise quality flags (STATUS → quality_flag, lowercased) and metric type (UOM).
    5. Drop unused metadata columns (symbol, terminated, decimals, uom_id, scalar_id, coordinate).
    6. Normalise GEO separators and collapse internal whitespace in INDICATOR.
    7. If the dataset contains Percent / Low 95% / High 95% characteristics, pivot them
       into value, ci_low, ci_high columns using a vectorised np.select + pivot_table
       approach (O(n log n)).
    8. Reorder columns for readability.

    Returns
    -------
    pd.DataFrame
        Cleaned tidy DataFrame ready for analysis or export.
    """
    df = df.copy()

    # --- 1. Normalise column names -------------------------------------------
    df.columns = df.columns.str.strip().str.lower()
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].str.strip()

    rename_map = {
        "age group": "age_group", "indicators": "indicator", "characteristics": "characteristic",
        "statistics": "statistic", "ref_date": "ref_date", "value": "value", "dguid": "dguid",
        "uom_id": "uom_id", "scalar_factor": "scalar_factor", "scalar_id": "scalar_id",
        "vector": "vector", "coordinate": "coordinate", "status": "quality_flag",
    }
    df.rename(columns=rename_map, inplace=True)
    if "gender" in df.columns:
        df.rename(columns={"gender": "sex"}, inplace=True)

    # --- 2. Parse reference dates --------------------------------------------
    df["ref_date_raw"] = df["ref_date"].astype(str).str.strip()
    df["start_year"] = pd.to_numeric(
        df["ref_date_raw"].str.extract(r"^(\d{4})")[0], errors="coerce"
    ).astype("Int64")

    # --- 3. Numeric values and scalar factors --------------------------------
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if "scalar_factor" in df.columns and "value" in df.columns:
        df.loc[df["scalar_factor"].eq("thousands"), "value"] *= 1000
        df["scalar_applied"] = df["scalar_factor"].eq("thousands")

    # --- 4. Quality flags and metric type ------------------------------------
    if "quality_flag" in df.columns:
        df["quality_flag"] = df["quality_flag"].str.lower().str.strip()
    if "uom" in df.columns:
        df["metric_type"] = df["uom"].str.lower()

    # --- 5. Drop unused metadata columns -------------------------------------
    drop_cols = ["symbol", "terminated", "decimals", "uom_id", "scalar_id", "coordinate"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    # --- 6. Normalise geography and indicator text ---------------------------
    if "geo" in df.columns:
        df["geo"] = df["geo"].str.replace(" / ", "/", regex=False).str.strip()
    if "indicator" in df.columns:
        df["indicator"] = df["indicator"].str.replace(r"\s+", " ", regex=True).str.strip()

    # --- 7. Vectorised pivot: Percent / CI rows → value / ci_low / ci_high --
    if "characteristic" in df.columns:
        has_percent = df["characteristic"].str.contains("Percent",  case=False, na=False).any()
        has_ci_low  = df["characteristic"].str.contains("Low.*95%", case=False, na=False).any()
        has_ci_high = df["characteristic"].str.contains("High.*95%",case=False, na=False).any()

        if has_percent and (has_ci_low or has_ci_high):
            # Classify each row by characteristic type in a single O(n) pass
            char = df["characteristic"].fillna("")
            df["_char_key"] = np.select(
                [
                    char.str.contains("Percent",  case=False),
                    char.str.contains("Low.*95%", case=False),
                    char.str.contains("High.*95%",case=False),
                ],
                ["value", "ci_low", "ci_high"],
                default="other",
            )

            # Dimension columns are everything that is not a value/metadata column
            meta_cols = {"value", "characteristic", "_char_key",
                         "metric_type", "quality_flag", "scalar_applied"}
            dim_cols  = [c for c in df.columns if c not in meta_cols]

            # Carry quality_flag and metric_type from the Percent rows into the pivot
            extra_keep = [c for c in ["quality_flag", "metric_type"] if c in df.columns]
            if extra_keep:
                pct_meta = (
                    df[df["_char_key"] == "value"][dim_cols + extra_keep]
                    .drop_duplicates(subset=dim_cols)
                )
            else:
                pct_meta = None

            # Pivot: one output column per characteristic key — O(n log n)
            pivot = (
                df[df["_char_key"] != "other"]
                .pivot_table(
                    index=dim_cols,
                    columns="_char_key",
                    values="value",
                    aggfunc="first",
                )
                .reset_index()
            )
            pivot.columns.name = None

            # Merge quality_flag / metric_type back from the Percent rows
            if pct_meta is not None and not pct_meta.empty:
                merge_cols  = [c for c in dim_cols if c in pct_meta.columns]
                merge_extra = [c for c in extra_keep if c in pct_meta.columns]
                pivot = pivot.merge(
                    pct_meta[merge_cols + merge_extra],
                    on=merge_cols,
                    how="left",
                )

            df = pivot

    # --- 8. Reorder columns for readability ----------------------------------
    col_order = [
        "ref_date_raw", "start_year", "geo", "sex", "age_group", "indicator",
        "characteristic", "value", "ci_low", "ci_high", "metric_type",
        "quality_flag", "scalar_applied", "vector",
    ]
    cols_present = [c for c in col_order if c in df.columns]
    other_cols   = [c for c in df.columns if c not in cols_present]
    df = df[cols_present + other_cols]

    return df


In [5]:
def clean_cihi_vizconfig(df: pd.DataFrame) -> pd.DataFrame:
    """Unpivot a CIHI chart-config CSV into one tidy row per data point.

    Each source row encodes multiple (x, y) pairs as comma-separated strings in
    x_axis_values and y_axis_values. This function splits those strings and returns
    one row per pair with columns: indicator, breakdown, group, value, ci_low, ci_high.

    Column presence is validated before processing; a descriptive ValueError is raised
    if required columns are missing.
    """
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()

    # Validate required columns before processing
    required = {"x_axis_values", "y_axis_values"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(
            f"clean_cihi_vizconfig: expected columns missing: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    df["x_axis_values"] = df["x_axis_values"].astype(str).str.split(",")
    df["y_axis_values"] = df["y_axis_values"].astype(str).str.split(",")

    tidy_rows = []
    for _, row in df.iterrows():
        indicator = row.get("indicator", "")
        xs = row["x_axis_values"]
        ys = row["y_axis_values"]
        n  = min(len(xs), len(ys))

        for i in range(n):
            x     = xs[i].strip()
            y     = ys[i].strip()
            value = pd.to_numeric(y, errors="coerce")
            tidy_rows.append({
                "indicator": indicator,
                "breakdown": row.get("vis_option", None),
                "group"    : x,
                "value"    : value,
                "ci_low"   : pd.to_numeric(row.get("confidence_interval_low",  None), errors="coerce"),
                "ci_high"  : pd.to_numeric(row.get("confidence_interval_high", None), errors="coerce"),
            })

    tidy = pd.DataFrame(tidy_rows)
    tidy = tidy.dropna(subset=["value"])
    return tidy


In [6]:
# Module-level constants for CI string parsing
_CI_DASH_CHARS = {"-", "\u2013", "\u2014"}       # ASCII hyphen, en-dash, em-dash
_CI_PATTERN    = r"([\d.]+)\s*[-\u2013\u2014]\s*([\d.]+)"  # matches "12.3-14.5" and variants


def clean_cihi_children_youth(raw_excel: dict) -> pd.DataFrame:
    """Reshape the CIHI children/youth Excel workbook from wide to long format.

    Parameters
    ----------
    raw_excel : dict[str, pd.DataFrame]
        Mapping of sheet name to raw DataFrame, as produced by the §3 loader.

    Returns
    -------
    pd.DataFrame
        Combined long-format DataFrame with columns:
        id columns | fiscal_year | value | ci_low | ci_high | sheet_type

    Notes
    -----
    - Year columns are detected dynamically via regex so future fiscal years are
      included automatically without code changes.
    - Cells formatted as CI ranges (e.g. '12.3-14.5') are split into ci_low and
      ci_high; the lower bound is written back as the point estimate in value.
    - All three dash variants (hyphen, en-dash, em-dash) are recognised.
    """
    frames = []

    for sheet, df in raw_excel.items():
        df = df.copy()
        df.columns = df.columns.str.strip().str.replace("\n", " ")

        # Detect year columns dynamically (any column whose name contains a 4-digit year)
        year_cols  = [c for c in df.columns if re.search(r"\b\d{4}\b", str(c))]
        id_cols    = [c for c in df.columns if c not in year_cols] if year_cols else list(df.columns[:3])
        value_cols = year_cols if year_cols else list(df.columns[3:])

        # Melt from wide (one column per fiscal year) to long format
        long_df = df.melt(
            id_vars=id_cols,
            value_vars=value_cols,
            var_name="fiscal_year",
            value_name="value",
        )
        long_df["fiscal_year"] = long_df["fiscal_year"].str.replace(" ", "", regex=False)

        # Reset index to ensure label-safe assignment with .loc below
        long_df = long_df.reset_index(drop=True)

        long_df["ci_low"]  = None
        long_df["ci_high"] = None

        # Parse CI range strings (e.g. '12.3-14.5') into separate bounds.
        # .items() is used so idx is the real label index, not a counter,
        # ensuring .loc[idx] always writes to the correct row.
        for idx, val in long_df["value"].items():
            if isinstance(val, str) and any(d in val for d in _CI_DASH_CHARS):
                match = pd.Series(val).str.extract(_CI_PATTERN, expand=True)
                if not match.empty and pd.notna(match.iloc[0, 0]):
                    lo = pd.to_numeric(match.iloc[0, 0], errors="coerce")
                    hi = pd.to_numeric(match.iloc[0, 1], errors="coerce")
                    long_df.loc[idx, "ci_low"]  = lo
                    long_df.loc[idx, "ci_high"] = hi
                    # Replace the raw CI string with the numeric point estimate
                    long_df.loc[idx, "value"]   = lo

        # Convert value column to numeric (CI strings are already resolved above)
        long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")

        # Assign a human-readable sheet type identifier
        long_df["sheet_type"] = (
            "ED"              if "Table8"  in sheet else
            "Hospitalization" if "Table13" in sheet else
            sheet.replace("_to hide", "")
        )
        frames.append(long_df)

    return pd.concat(frames, ignore_index=True)


In [7]:
def clean_mhacs_pumf(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the MHACS 2022 Public Use Microdata File.

    Replaces StatCan non-response codes with NaN for all numeric D-prefix variables.
    Target columns are derived at runtime from the DataFrame so this function
    remains correct if the file gains new D-prefix variables in future releases.

    Missing-value codes replaced
    ----------------------------
    6   Not applicable
    7   Don't know
    8   Refused
    9   Not stated
    96  Not applicable (extended)
    996 Not applicable (extended)
    999 Not applicable (extended)
    99.6 Continuous not stated
    """
    df = df.copy()

    # All numeric columns whose name starts with 'D' — covers the 43 confirmed
    # D-prefix variables in the MHACS 2022 PUMF and any additions in future releases
    target_vars = [
        col for col in df.columns
        if col.startswith("D") and df[col].dtype in ("int64", "float64")
    ]

    missing_codes = {6, 7, 8, 9, 96, 996, 999, 99.6}
    for col in target_vars:
        df[col] = df[col].replace(list(missing_codes), np.nan)

    # Ensure the survey weight is numeric
    if "WTS_M" in df.columns:
        df["WTS_M"] = pd.to_numeric(df["WTS_M"], errors="coerce")

    return df


## §4·FIX — `clean_statcan_long`
**Patch for `notebooks/02_data_cleaning.ipynb` (cell §4) and `notebooks/02_data_cleaning_functions.py`.**

The original pivot produced duplicate-grain rows (e.g. `stress_coping.csv`: 14,388 rows / 11,510 duplicates) and put percent-CI bounds in the `value` column. Two causes:

1. `vector` (unique per row) was in the pivot index → the point / CI-low / CI-high rows never grouped.
2. `"Low 95% … percent"` contains the substring *percent*, so it was labelled as the point estimate.

Run this cell **after §4** and **before §5**. Verified on all 5 StatCan tables: 0 duplicate grain, `value` always within `[ci_low, ci_high]`. Once the pipeline output looks right, fold this body back into the §4 cell + the `.py`, and delete this cell.

In [8]:
# =====================================================================
# §4·FIX  —  clean_statcan_long
# Patch for:  notebooks/02_data_cleaning.ipynb  (cell §4)
#             notebooks/02_data_cleaning_functions.py
# =====================================================================
# WHY: the previous pivot left `vector` (unique per row) in the pivot
#      index, so the point / CI-low / CI-high rows never grouped
#      -> duplicate grain (stress_coping had 11,510 dup rows), and it
#      classified "Low/High 95% ... percent" as the point estimate
#      because those labels contain the substring "percent".
# WHAT THIS CELL DOES: redefines clean_statcan_long. Run it AFTER the
#      §4 cells and BEFORE §5 (Run pipeline). "Restart & Run All" also
#      picks it up because it is the last definition.
# VERIFIED 2026-09-02 on all 5 StatCan tables: 0 duplicate grain,
#      value always within [ci_low, ci_high].
# When happy, paste this body over the §4 clean_statcan_long cell and
# over the same function in 02_data_cleaning_functions.py, then delete
# this cell.
# =====================================================================

def clean_statcan_long(df: pd.DataFrame) -> pd.DataFrame:
    """Clean a StatCan 'table download' CSV into one tidy row per
    geo x period x sex x age_group x indicator, with value / ci_low / ci_high
    as columns (percentage metric)."""
    df = df.copy()

    # --- column names: strip + lowercase, strip text cells ---
    df.columns = df.columns.str.strip().str.lower()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    rename_map = {
        "age group": "age_group", "indicators": "indicator",
        "characteristics": "characteristic", "statistics": "characteristic",
        "value": "value", "status": "quality_flag",
    }
    df.rename(columns=rename_map, inplace=True)
    if "gender" in df.columns:
        df.rename(columns={"gender": "sex"}, inplace=True)

    # --- dates: keep original, derive numeric start year ---
    df["ref_date_raw"] = df["ref_date"].astype(str).str.strip()
    df["start_year"] = pd.to_numeric(
        df["ref_date_raw"].str.extract(r"^(\d{4})")[0], errors="coerce"
    ).astype("Int64")

    # --- value -> numeric, apply scalar factor ---
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if "scalar_factor" in df.columns:
        df.loc[df["scalar_factor"].eq("thousands"), "value"] *= 1000

    # --- normalise text ---
    if "geo" in df.columns:
        df["geo"] = df["geo"].str.replace(r"\s+", " ", regex=True).str.strip()
    if "indicator" in df.columns:
        df["indicator"] = df["indicator"].str.replace(r"\s+", " ", regex=True).str.strip()
    if "quality_flag" in df.columns:
        df["quality_flag"] = (df["quality_flag"].astype(str).str.strip()
                              .str.upper().replace({"NAN": ""}))

    # --- pivot the characteristic dimension into value / ci_low / ci_high ---
    if "characteristic" in df.columns:
        char = df["characteristic"].fillna("").str.lower()

        # FIX 2: the label always says the metric ("percent" vs count)
        df["_metric"] = np.where(char.str.contains("percent"), "percent", "number")

        df["_stat"] = np.select(
            [
                char.str.contains(r"(?:low|lower|under).*95%",   regex=True),
                char.str.contains(r"(?:high|upper|higher).*95%", regex=True),
                char.str.contains("coefficient of variation"),
                char.str.contains("statistically different"),
            ],
            ["ci_low", "ci_high", "cv", "sig_diff"],
            default="value",
        )

        has_ci = df["_stat"].isin(["ci_low", "ci_high"]).any()
        if has_ci:
            keep = df[(df["_metric"] == "percent")
                      & (df["_stat"].isin(["value", "ci_low", "ci_high"]))].copy()

            # FIX 1: ONLY true dimensions in the index (never vector / dguid / ref_date)
            dim_cols = [c for c in ["ref_date_raw", "start_year", "geo",
                                    "sex", "age_group", "indicator"] if c in keep.columns]

            meta_cols = ["quality_flag"] if "quality_flag" in keep.columns else []
            meta = (keep[keep["_stat"] == "value"][dim_cols + meta_cols]
                    .drop_duplicates(subset=dim_cols)) if meta_cols else None

            pivot = (keep.pivot_table(index=dim_cols, columns="_stat",
                                      values="value", aggfunc="first")
                         .reset_index())
            pivot.columns.name = None
            for c in ["value", "ci_low", "ci_high"]:
                if c not in pivot.columns:
                    pivot[c] = np.nan
            if meta is not None:
                pivot = pivot.merge(meta, on=dim_cols, how="left")
            pivot["metric_type"] = "percent"
            df = pivot
        else:
            # no CI rows (e.g. perceived_mh_annual): keep point rows, tag metric
            df = df[df["_stat"] == "value"].copy()
            df["metric_type"] = df["_metric"]
            df["ci_low"] = np.nan
            df["ci_high"] = np.nan

    # --- drop leftover metadata columns ---
    df = df.drop(columns=[c for c in ["vector", "dguid", "ref_date", "coordinate",
                                      "uom", "uom_id", "scalar_id", "scalar_factor",
                                      "symbol", "terminated", "decimals",
                                      "characteristic", "_metric", "_stat"]
                          if c in df.columns], errors="ignore")

    order = ["ref_date_raw", "start_year", "geo", "sex", "age_group", "indicator",
             "value", "ci_low", "ci_high", "metric_type", "quality_flag"]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    return df[cols].reset_index(drop=True)


# quick self-check on the loaded raw StatCan tables
for _k, _spec in DATASETS.items():
    if _spec["kind"] != "statcan_long":
        continue
    _o = clean_statcan_long(raw[_k])
    _grain = [c for c in ["ref_date_raw", "geo", "sex", "age_group", "indicator"]
              if c in _o.columns]
    _dups = _o.duplicated(subset=_grain).sum()
    print(f"{_k:28} clean={_o.shape[0]:>6} rows  dup_grain={_dups}")


perceived_mh_annual          clean=   936 rows  dup_grain=0
suicidal_thoughts            clean=  1104 rows  dup_grain=0


stress_coping                clean=  3220 rows  dup_grain=0
perceived_health_quarterly   clean=  1053 rows  dup_grain=0


cchs_mh_disorders            clean= 12917 rows  dup_grain=0


## §5 · Run pipeline

Applies the four cleaning functions to every dataset and writes the results to `data/processed/02_cleaned/`. Each output file is named after its dataset key (e.g. `perceived_mh_annual.csv`).


In [9]:
CLEANED = ROOT / "data" / "processed" / "02_cleaned"
CLEANED.mkdir(parents=True, exist_ok=True)

cleaned: dict[str, pd.DataFrame] = {}

for key, spec in DATASETS.items():
    print(f"\n=== Cleaning {key} ===")

    if spec["kind"] == "statcan_long":
        df = clean_statcan_long(raw[key])

    elif spec["kind"] == "cihi_vizconfig":
        df = clean_cihi_vizconfig(raw[key])

    elif spec["kind"] == "excel_multitable":
        df = clean_cihi_children_youth(raw_excel)

    elif spec["kind"] == "microdata":
        df = clean_mhacs_pumf(raw[key])

    else:
        print(f"  Skipped: no cleaner for kind={spec['kind']}")
        continue

    cleaned[key] = df
    df.to_csv(CLEANED / f"{key}.csv", index=False)
    print(f"  Written: {CLEANED / f'{key}.csv'}  shape={df.shape}")

print("\nALL DATASETS CLEANED ✔")



=== Cleaning perceived_mh_annual ===
  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/perceived_mh_annual.csv  shape=(936, 11)

=== Cleaning suicidal_thoughts ===
  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/suicidal_thoughts.csv  shape=(1104, 11)

=== Cleaning stress_coping ===


  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/stress_coping.csv  shape=(3220, 11)

=== Cleaning perceived_health_quarterly ===
  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/perceived_health_quarterly.csv  shape=(1053, 10)

=== Cleaning cchs_mh_disorders ===


  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/cchs_mh_disorders.csv  shape=(12917, 11)

=== Cleaning cihi_mh_services ===
  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/cihi_mh_services.csv  shape=(263, 6)

=== Cleaning cihi_children_youth ===


  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/cihi_children_youth.csv  shape=(4320, 8)

=== Cleaning mhacs_2022_pumf ===


  Written: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed/02_cleaned/mhacs_2022_pumf.csv  shape=(9861, 602)

ALL DATASETS CLEANED ✔
